<a href="https://colab.research.google.com/github/mbetagonza/MentorIA-Lab/blob/main/Simulador_2D_ultrasonido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip -q install gradio scipy scikit-image matplotlib

In [2]:
# =============================================================
# Simulador 2D de ultrasonido pulse-echo para Google Colab + Gradio
# Versión con 3 niveles de simulación inspirados en SIMUS
#
# Nivel 1: Reflectividad continua R(x,z) + campo de transductor por elementos
#          r_l(t) = K ∫∫ R(x,z) n(t-2z/c0) A(z) |q_l(x,z)|² dx dz
#
# Nivel 2: Reflectividad continua discretizada en puntos de grilla + RF por canal
#          + beamforming delay-and-sum.
#
# Nivel 3: Scatterers puntuales aleatorios estilo SIMUS simplificado + RF por canal
#          + beamforming delay-and-sum.
#
# En Google Colab, ejecutar primero:
# !pip -q install gradio scipy scikit-image matplotlib
# =============================================================

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from scipy.ndimage import gaussian_filter
from scipy.signal import hilbert, fftconvolve
from skimage.draw import disk


# =============================================================
# 0. Utilidades generales
# =============================================================

def robust_norm(img, pmin=1, pmax=99):
    img = np.asarray(img, dtype=np.float64)
    lo, hi = np.percentile(img, [pmin, pmax])
    out = (img - lo) / (hi - lo + 1e-12)
    return np.clip(out, 0, 1)


def make_fig_image(img, title="", cmap="gray", extent=None, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(5.4, 4.2), dpi=130)
    ax.imshow(img, cmap=cmap, aspect="auto", extent=extent, origin="upper", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    if extent is not None:
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("z [mm]")
    ax.grid(False)
    fig.tight_layout()
    return fig


def make_line_fig(x, y, title="", xlabel="", ylabel=""):
    fig, ax = plt.subplots(figsize=(5.5, 3.5), dpi=130)
    ax.plot(x, y)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


def sincu(u):
    """sinc no normalizada: sin(u)/u."""
    return np.sinc(u / np.pi)


def interp1_uniform(y, x_float):
    """Interpolación lineal rápida sobre una señal 1D y[n] en posiciones flotantes."""
    n = len(y)
    x0 = np.floor(x_float).astype(int)
    f = x_float - x0
    valid = (x0 >= 0) & (x0 < n - 1)
    out = np.zeros_like(x_float, dtype=np.float64)
    out[valid] = (1 - f[valid]) * y[x0[valid]] + f[valid] * y[x0[valid] + 1]
    return out


# =============================================================
# 1. Phantom: c(x,z), rho(x,z), impedancia y reflectividad
# =============================================================

def generate_phantom(
    nx=256,
    nz=384,
    width_mm=40.0,
    depth_mm=70.0,
    c_bg=1540.0,
    rho_bg=1000.0,
    c_inc1=1480.0,
    rho_inc1=950.0,
    inc1_x_mm=-8.0,
    inc1_z_mm=32.0,
    inc1_r_mm=6.0,
    c_inc2=1600.0,
    rho_inc2=1060.0,
    inc2_x_mm=9.0,
    inc2_z_mm=47.0,
    inc2_r_mm=8.0,
    speckle_strength=0.025,
    seed=1,
    smooth_sigma=0.7,
):
    """Genera un phantom 2D con fondo y dos inclusiones.

    c(x,z): velocidad del sonido [m/s]
    rho(x,z): densidad [kg/m^3]
    Z(x,z)=rho*c: impedancia acústica
    R_edges(x,z)=|∇logZ|: reflectividad de interfaces
    R(x,z)=R_edges + microdispersión: reflectividad total usada por el simulador
    """
    rng = np.random.default_rng(int(seed))

    x_mm = np.linspace(-width_mm / 2, width_mm / 2, int(nx))
    z_mm = np.linspace(0, depth_mm, int(nz))
    dx_m = (x_mm[1] - x_mm[0]) * 1e-3
    dz_m = (z_mm[1] - z_mm[0]) * 1e-3

    c_map = np.full((int(nz), int(nx)), c_bg, dtype=np.float64)
    rho_map = np.full((int(nz), int(nx)), rho_bg, dtype=np.float64)

    def add_circle(c_val, rho_val, x0_mm, z0_mm, r_mm):
        ix = int(np.argmin(np.abs(x_mm - x0_mm)))
        iz = int(np.argmin(np.abs(z_mm - z0_mm)))
        rx_pix = max(1, int(r_mm / (x_mm[1] - x_mm[0])))
        rz_pix = max(1, int(r_mm / (z_mm[1] - z_mm[0])))
        r_pix = int(0.5 * (rx_pix + rz_pix))
        rr, cc = disk((iz, ix), r_pix, shape=c_map.shape)
        c_map[rr, cc] = c_val
        rho_map[rr, cc] = rho_val

    add_circle(c_inc1, rho_inc1, inc1_x_mm, inc1_z_mm, inc1_r_mm)
    add_circle(c_inc2, rho_inc2, inc2_x_mm, inc2_z_mm, inc2_r_mm)

    if smooth_sigma > 0:
        c_map = gaussian_filter(c_map, smooth_sigma)
        rho_map = gaussian_filter(rho_map, smooth_sigma)

    Z_map = rho_map * c_map
    logZ = np.log(Z_map + 1e-12)

    gz, gx = np.gradient(logZ)
    R_edges = np.sqrt(gx**2 + gz**2)
    R_edges = R_edges / (np.percentile(R_edges, 99.5) + 1e-12)
    R_edges = np.clip(R_edges, 0, 1)

    # Microdispersores: componente granular que permite speckle en los niveles 2 y 3.
    micro = rng.normal(0.0, 1.0, size=(int(nz), int(nx)))
    micro = gaussian_filter(micro, 0.35)
    micro = micro / (np.std(micro) + 1e-12)

    R_total = R_edges + speckle_strength * micro
    R_total = R_total - np.mean(R_total)

    grids = {
        "x_mm": x_mm,
        "z_mm": z_mm,
        "dx_m": dx_m,
        "dz_m": dz_m,
        "width_mm": width_mm,
        "depth_mm": depth_mm,
    }

    return {
        "c": c_map,
        "rho": rho_map,
        "Z": Z_map,
        "R_edges": R_edges,
        "R": R_total,
        "micro": micro,
        "grids": grids,
    }


# =============================================================
# 2. Transductor y pulso
# =============================================================

def gaussian_pulse(f0_mhz=5.0, cycles=2.5, dt=2e-7, duration_factor=4.0):
    """Pulso RF n(t): senoide modulada por una gaussiana."""
    f0 = f0_mhz * 1e6
    sigma_t = cycles / (2.5 * f0)
    t_max = duration_factor * sigma_t
    t = np.arange(-t_max, t_max + dt, dt)
    env = np.exp(-0.5 * (t / sigma_t) ** 2)
    pulse = env * np.cos(2 * np.pi * f0 * t)
    pulse -= np.mean(pulse)
    pulse /= np.max(np.abs(pulse)) + 1e-12
    return t, pulse


def get_element_positions(n_elements=64, pitch_mm=0.3):
    n_elements = int(n_elements)
    idx = np.arange(n_elements) - (n_elements - 1) / 2
    return idx * pitch_mm


def apodization_for_line(element_x_mm, line_x_mm, aperture_mm, kind="hann"):
    """Apodización de transmisión o recepción centrada en una línea."""
    element_x_mm = np.asarray(element_x_mm)
    half = max(aperture_mm / 2, 1e-6)
    u = (element_x_mm - line_x_mm) / half
    active = np.abs(u) <= 1
    w = np.zeros_like(element_x_mm, dtype=np.float64)

    if np.any(active):
        if kind == "rect":
            w[active] = 1.0
        else:
            # Hann centrada: vale 1 al centro y 0 en los bordes de la apertura.
            w[active] = 0.5 * (1 + np.cos(np.pi * u[active]))
    else:
        # Si la línea cae fuera de la apertura, al menos activamos el elemento más cercano.
        w[np.argmin(np.abs(element_x_mm - line_x_mm))] = 1.0

    if np.sum(w) > 0:
        w = w / (np.max(w) + 1e-12)
    return w


def compute_gaussian_beam_map(x_mm, z_mm, line_x_mm, f0_mhz=5.0, aperture_mm=16.0, focus_mm=35.0):
    """Haz gaussiano pedagógico usado como referencia rápida."""
    c0 = 1540.0
    lam_mm = c0 / (f0_mhz * 1e6) * 1e3
    f_number = max(focus_mm / max(aperture_mm, 0.5), 0.1)
    sigma_focus = max(1.2 * lam_mm * f_number, 0.25)
    divergence = lam_mm / max(aperture_mm, 0.5) * 2.2
    sigma_z = np.sqrt(sigma_focus**2 + (divergence * (z_mm - focus_mm))**2)

    X, Z = np.meshgrid(x_mm, z_mm)
    SIG = sigma_z[:, None]
    q_amp = np.exp(-0.5 * ((X - line_x_mm) / (SIG + 1e-12)) ** 2)
    axial_focus = np.exp(-0.5 * ((Z - focus_mm) / (0.85 * focus_mm + 1e-12)) ** 2)
    q_amp = q_amp * (0.35 + 0.65 * axial_focus)
    q2 = q_amp**2
    return q2 / (np.max(q2) + 1e-12)


def array_tx_field_points(
    xs_mm,
    zs_mm,
    line_x_mm,
    f0_mhz=5.0,
    n_elements=64,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=16.0,
    focus_mm=35.0,
    c0=1540.0,
    apod_kind="hann",
    soft_baffle=True,
):
    """Campo transmitido complejo q_tx evaluado en puntos.

    Modelo 2D simplificado inspirado en SIMUS:
    q_tx(X,omega) = Σ_n W_n D(theta_n,k) exp(i k (r_n-r_focus_n)) / sqrt(r_n)

    Se usa 1/sqrt(r) porque es una aproximación acústica 2D.
    """
    xs_m = np.asarray(xs_mm, dtype=np.float64) * 1e-3
    zs_m = np.asarray(zs_mm, dtype=np.float64) * 1e-3
    xe_mm = get_element_positions(n_elements, pitch_mm)
    xe_m = xe_mm * 1e-3

    f0 = f0_mhz * 1e6
    k = 2 * np.pi * f0 / c0
    b_m = max(element_width_mm, 0.01) * 1e-3 / 2
    zf_m = max(focus_mm, 1.0) * 1e-3
    xf_m = line_x_mm * 1e-3

    w = apodization_for_line(xe_mm, line_x_mm, aperture_mm, apod_kind)

    q = np.zeros(xs_m.shape, dtype=np.complex128)
    for xe, wi in zip(xe_m, w):
        if wi == 0:
            continue
        r = np.sqrt((xs_m - xe)**2 + zs_m**2) + 1e-12
        r_focus = np.sqrt((xf_m - xe)**2 + zf_m**2) + 1e-12
        sin_theta = (xs_m - xe) / r
        cos_theta = np.maximum(zs_m / r, 0)
        D = sincu(k * b_m * sin_theta)
        if soft_baffle:
            D = D * cos_theta
        q += wi * D * np.exp(1j * k * (r - r_focus)) / np.sqrt(r)

    # Normalización de amplitud para evitar que cambie drásticamente con N.
    amp = np.abs(q)
    q = q / (np.percentile(amp, 99.5) + 1e-12)
    return q


def compute_array_beam_map(
    x_mm,
    z_mm,
    line_x_mm,
    f0_mhz=5.0,
    n_elements=64,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=16.0,
    focus_mm=35.0,
    c0=1540.0,
):
    X, Z = np.meshgrid(x_mm, z_mm)
    q = array_tx_field_points(
        X.ravel(), Z.ravel(), line_x_mm,
        f0_mhz=f0_mhz,
        n_elements=n_elements,
        pitch_mm=pitch_mm,
        element_width_mm=element_width_mm,
        aperture_mm=aperture_mm,
        focus_mm=focus_mm,
        c0=c0,
    )
    q2 = np.abs(q.reshape(len(z_mm), len(x_mm)))**2
    return q2 / (np.max(q2) + 1e-12)


def rx_element_sensitivity_points(xs_mm, zs_mm, element_x_mm, f0_mhz=5.0, element_width_mm=0.24, c0=1540.0, soft_baffle=True):
    xs_m = np.asarray(xs_mm) * 1e-3
    zs_m = np.asarray(zs_mm) * 1e-3
    xe_m = element_x_mm * 1e-3
    f0 = f0_mhz * 1e6
    k = 2 * np.pi * f0 / c0
    b_m = max(element_width_mm, 0.01) * 1e-3 / 2
    r = np.sqrt((xs_m - xe_m)**2 + zs_m**2) + 1e-12
    sin_theta = (xs_m - xe_m) / r
    cos_theta = np.maximum(zs_m / r, 0)
    D = sincu(k * b_m * sin_theta)
    if soft_baffle:
        D = D * cos_theta
    return D / np.sqrt(r)


# =============================================================
# 3. Atenuación
# =============================================================

def attenuation_profile(z_mm, mu_db_cm_mhz=0.5, f0_mhz=5.0):
    """Atenuación round-trip en amplitud.

    alpha_db/cm = mu_db_cm_mhz * f0_mhz
    factor = 10^[-(2 alpha z_cm)/20]
    """
    z_cm = np.asarray(z_mm) / 10.0
    alpha_db_cm = mu_db_cm_mhz * f0_mhz
    return 10 ** (-(2.0 * alpha_db_cm * z_cm) / 20.0)


# =============================================================
# 4A. Nivel 1: reflectividad continua + q por elementos
# =============================================================

def simulate_level1_continuous(
    phantom,
    n_lines=96,
    selected_line=48,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=64,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=16.0,
    focus_mm=35.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    dynamic_range_db=55.0,
    beam_kind="Elementos por difracción",
):
    R = phantom["R"]
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    dx_m = grids["dx_m"]
    dz_m = grids["dz_m"]
    nz, nx = R.shape

    c0 = float(np.mean(phantom["c"]))
    dt_depth = 2.0 * dz_m / c0
    _, pulse = gaussian_pulse(f0_mhz=f0_mhz, cycles=cycles, dt=dt_depth)

    line_positions = np.linspace(x_mm.min(), x_mm.max(), int(n_lines))
    RF = np.zeros((nz, int(n_lines)), dtype=np.float64)
    pre_echo = np.zeros_like(RF)
    atten = attenuation_profile(z_mm, mu_db_cm_mhz=mu_db_cm_mhz, f0_mhz=f0_mhz)

    for li, x0 in enumerate(line_positions):
        if beam_kind.startswith("Gaussiano"):
            q2 = compute_gaussian_beam_map(x_mm, z_mm, x0, f0_mhz, aperture_mm, focus_mm)
        else:
            q2 = compute_array_beam_map(
                x_mm, z_mm, x0,
                f0_mhz=f0_mhz,
                n_elements=n_elements,
                pitch_mm=pitch_mm,
                element_width_mm=element_width_mm,
                aperture_mm=aperture_mm,
                focus_mm=focus_mm,
                c0=c0,
            )

        A_z = K * np.sum(R * q2, axis=1) * dx_m
        A_z = A_z * atten
        pre_echo[:, li] = A_z
        RF[:, li] = fftconvolve(A_z, pulse, mode="same") * dz_m

    ENV = np.abs(hilbert(RF, axis=0))
    ENV /= np.max(ENV) + 1e-12
    B_db = 20.0 * np.log10(ENV + 1e-6)
    B_norm = np.clip((B_db + dynamic_range_db) / dynamic_range_db, 0, 1)

    selected_line = int(np.clip(selected_line, 0, int(n_lines) - 1))
    if beam_kind.startswith("Gaussiano"):
        q2_sel = compute_gaussian_beam_map(x_mm, z_mm, line_positions[selected_line], f0_mhz, aperture_mm, focus_mm)
    else:
        q2_sel = compute_array_beam_map(
            x_mm, z_mm, line_positions[selected_line],
            f0_mhz=f0_mhz,
            n_elements=n_elements,
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            c0=c0,
        )

    return {
        "mode": "Nivel 1: continuo + q por elementos",
        "RF": RF,
        "ENV": ENV,
        "B": B_norm,
        "B_db": B_db,
        "A_mode": ENV[:, selected_line],
        "RF_line": RF[:, selected_line],
        "pre_line": pre_echo[:, selected_line],
        "q2_selected": q2_sel,
        "line_positions": line_positions,
        "z_mm": z_mm,
        "x_mm": x_mm,
        "selected_line": selected_line,
        "dt": dt_depth,
        "n_points": int(nx * nz),
    }


# =============================================================
# 4B. Puntos discretos: grilla regular o scatterers aleatorios
# =============================================================

def regular_grid_points_from_reflectivity(phantom, stride=3, max_points=8000):
    """Convierte R(x,z) en puntos regulares de grilla.

    Esto es parecido a tratar cada celda como un scatterer fijo en el centro del pixel.
    """
    R = phantom["R"]
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    dx_m = grids["dx_m"]
    dz_m = grids["dz_m"]

    zz_idx = np.arange(0, R.shape[0], int(stride))
    xx_idx = np.arange(0, R.shape[1], int(stride))
    ZZ, XX = np.meshgrid(zz_idx, xx_idx, indexing="ij")
    xs = x_mm[XX.ravel()]
    zs = z_mm[ZZ.ravel()]
    amps = R[ZZ.ravel(), XX.ravel()] * dx_m * dz_m * (stride**2)

    # Quitar puntos casi nulos para acelerar.
    keep = np.abs(amps) > np.percentile(np.abs(amps), 40)
    xs, zs, amps = xs[keep], zs[keep], amps[keep]

    if len(xs) > max_points:
        rng = np.random.default_rng(0)
        prob = np.abs(amps) + 1e-12
        prob = prob / np.sum(prob)
        idx = rng.choice(len(xs), size=int(max_points), replace=False, p=prob)
        xs, zs, amps = xs[idx], zs[idx], amps[idx]

    return xs, zs, amps


def scatterers_from_phantom(phantom, n_scatterers=6000, seed=1, interface_gain=1.5, speckle_gain=0.6):
    """Genera scatterers aleatorios a partir del phantom.

    La amplitud combina:
    - componente aleatoria de speckle
    - componente de interfaces desde R_edges
    """
    rng = np.random.default_rng(int(seed))
    grids = phantom["grids"]
    x_mm_grid = grids["x_mm"]
    z_mm_grid = grids["z_mm"]
    R_edges = phantom["R_edges"]
    micro = phantom["micro"]

    xs = rng.uniform(x_mm_grid.min(), x_mm_grid.max(), int(n_scatterers))
    zs = rng.uniform(max(0.2, z_mm_grid.min()), z_mm_grid.max(), int(n_scatterers))

    # Interpolación nearest para mantenerlo simple y rápido.
    ix = np.clip(np.searchsorted(x_mm_grid, xs), 1, len(x_mm_grid)-1)
    iz = np.clip(np.searchsorted(z_mm_grid, zs), 1, len(z_mm_grid)-1)
    ix = np.where(np.abs(x_mm_grid[ix] - xs) < np.abs(x_mm_grid[ix-1] - xs), ix, ix-1)
    iz = np.where(np.abs(z_mm_grid[iz] - zs) < np.abs(z_mm_grid[iz-1] - zs), iz, iz-1)

    interface_amp = R_edges[iz, ix]
    tissue_amp = np.abs(micro[iz, ix])

    # Coeficientes reales con signo aleatorio. La fase RF se generará por los retardos del pulso.
    signs = rng.choice([-1.0, 1.0], size=int(n_scatterers))
    amps = signs * (speckle_gain * (0.25 + tissue_amp) + interface_gain * interface_amp)
    amps = amps / (np.std(amps) + 1e-12)
    amps = amps * grids["dx_m"] * grids["dz_m"]
    return xs, zs, amps


# =============================================================
# 4C. Nivel 2/3: RF por canal + DAS
# =============================================================

def simulate_rf_channels_for_line(
    xs_mm,
    zs_mm,
    amps,
    phantom,
    line_x_mm,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=32,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=12.0,
    focus_mm=35.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    nt=None,
):
    """Genera RF por canal para una línea.

    Simplificación geométrica:
    - transmisión efectiva tipo línea vertical: tau_tx = z/c0
    - recepción por elemento: tau_rx_m = sqrt((x-x_m)^2+z^2)/c0
    - el campo transmitido q_tx sí se calcula por elementos para ponderar amplitud.
    """
    grids = phantom["grids"]
    z_mm_grid = grids["z_mm"]
    dz_m = grids["dz_m"]
    c0 = float(np.mean(phantom["c"]))
    dt = 2.0 * dz_m / c0
    if nt is None:
        nt = len(z_mm_grid)

    _, pulse = gaussian_pulse(f0_mhz=f0_mhz, cycles=cycles, dt=dt)
    element_x_mm = get_element_positions(n_elements, pitch_mm)
    rx_w = apodization_for_line(element_x_mm, line_x_mm, aperture_mm, kind="hann")

    # Campo transmitido en los puntos.
    qtx = array_tx_field_points(
        xs_mm, zs_mm, line_x_mm,
        f0_mhz=f0_mhz,
        n_elements=n_elements,
        pitch_mm=pitch_mm,
        element_width_mm=element_width_mm,
        aperture_mm=aperture_mm,
        focus_mm=focus_mm,
        c0=c0,
    )
    qtx_amp = np.abs(qtx)
    qtx_amp = qtx_amp / (np.percentile(qtx_amp, 99.5) + 1e-12)

    atten = attenuation_profile(zs_mm, mu_db_cm_mhz=mu_db_cm_mhz, f0_mhz=f0_mhz)

    xs_m = xs_mm * 1e-3
    zs_m = zs_mm * 1e-3
    RF_ch = np.zeros((int(nt), int(n_elements)), dtype=np.float64)

    tau_tx = zs_m / c0

    for mi, xe_mm in enumerate(element_x_mm):
        xe_m = xe_mm * 1e-3
        r_rx = np.sqrt((xs_m - xe_m)**2 + zs_m**2)
        tau = tau_tx + r_rx / c0
        idx = tau / dt
        i0 = np.floor(idx).astype(int)
        frac = idx - i0
        valid = (i0 >= 0) & (i0 < int(nt) - 1)

        qrx = rx_element_sensitivity_points(xs_mm, zs_mm, xe_mm, f0_mhz, element_width_mm, c0)
        qrx_amp = np.abs(qrx)
        qrx_amp = qrx_amp / (np.percentile(qrx_amp, 99.5) + 1e-12)

        impulse = np.zeros(int(nt), dtype=np.float64)
        val = K * amps * qtx_amp * qrx_amp * atten
        np.add.at(impulse, i0[valid], val[valid] * (1 - frac[valid]))
        np.add.at(impulse, i0[valid] + 1, val[valid] * frac[valid])

        RF_ch[:, mi] = fftconvolve(impulse, pulse, mode="same")

    return RF_ch, dt, element_x_mm


def das_beamform_line(RF_ch, z_mm, line_x_mm, element_x_mm, aperture_mm=12.0, c0=1540.0, dt=1e-7):
    """Delay-and-sum de recepción para una línea."""
    z_m = z_mm * 1e-3
    x0_m = line_x_mm * 1e-3
    rx_w = apodization_for_line(element_x_mm, line_x_mm, aperture_mm, kind="hann")

    b = np.zeros_like(z_mm, dtype=np.float64)
    for mi, xe_mm in enumerate(element_x_mm):
        xe_m = xe_mm * 1e-3
        tau = z_m / c0 + np.sqrt((x0_m - xe_m)**2 + z_m**2) / c0
        samples = tau / dt
        b += rx_w[mi] * interp1_uniform(RF_ch[:, mi], samples)

    b /= (np.sum(rx_w) + 1e-12)
    return b


def simulate_channel_das(
    phantom,
    level="Nivel 2: grilla continua discretizada + RF por canal + DAS",
    n_lines=64,
    selected_line=32,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=32,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=12.0,
    focus_mm=35.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    dynamic_range_db=55.0,
    grid_stride=3,
    n_scatterers=5000,
    scatter_seed=1,
    interface_gain=1.5,
    speckle_gain=0.6,
):
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    c0 = float(np.mean(phantom["c"]))
    nt = len(z_mm)

    if level.startswith("Nivel 3"):
        xs, zs, amps = scatterers_from_phantom(
            phantom,
            n_scatterers=int(n_scatterers),
            seed=int(scatter_seed),
            interface_gain=interface_gain,
            speckle_gain=speckle_gain,
        )
        mode_name = "Nivel 3: scatterers puntuales + RF por canal + DAS"
    else:
        xs, zs, amps = regular_grid_points_from_reflectivity(
            phantom,
            stride=int(grid_stride),
            max_points=int(n_scatterers),
        )
        mode_name = "Nivel 2: R(x,z) discretizada + RF por canal + DAS"

    line_positions = np.linspace(x_mm.min(), x_mm.max(), int(n_lines))
    RF_bf = np.zeros((nt, int(n_lines)), dtype=np.float64)
    selected_line = int(np.clip(selected_line, 0, int(n_lines) - 1))
    RF_ch_selected = None
    dt_selected = None
    element_x_selected = None

    for li, x0 in enumerate(line_positions):
        RF_ch, dt, element_x_mm = simulate_rf_channels_for_line(
            xs, zs, amps, phantom, x0,
            f0_mhz=f0_mhz,
            cycles=cycles,
            n_elements=n_elements,
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            mu_db_cm_mhz=mu_db_cm_mhz,
            K=K,
            nt=nt,
        )
        RF_bf[:, li] = das_beamform_line(
            RF_ch, z_mm, x0, element_x_mm,
            aperture_mm=aperture_mm,
            c0=c0,
            dt=dt,
        )
        if li == selected_line:
            RF_ch_selected = RF_ch
            dt_selected = dt
            element_x_selected = element_x_mm

    ENV = np.abs(hilbert(RF_bf, axis=0))
    ENV /= np.max(ENV) + 1e-12
    B_db = 20.0 * np.log10(ENV + 1e-6)
    B_norm = np.clip((B_db + dynamic_range_db) / dynamic_range_db, 0, 1)

    q2_sel = compute_array_beam_map(
        x_mm, z_mm, line_positions[selected_line],
        f0_mhz=f0_mhz,
        n_elements=n_elements,
        pitch_mm=pitch_mm,
        element_width_mm=element_width_mm,
        aperture_mm=aperture_mm,
        focus_mm=focus_mm,
        c0=c0,
    )

    return {
        "mode": mode_name,
        "RF": RF_bf,
        "ENV": ENV,
        "B": B_norm,
        "B_db": B_db,
        "A_mode": ENV[:, selected_line],
        "RF_line": RF_bf[:, selected_line],
        "RF_channels_selected": RF_ch_selected,
        "q2_selected": q2_sel,
        "line_positions": line_positions,
        "z_mm": z_mm,
        "x_mm": x_mm,
        "selected_line": selected_line,
        "dt": dt_selected,
        "element_x_mm": element_x_selected,
        "scatter_x_mm": xs,
        "scatter_z_mm": zs,
        "scatter_amp": amps,
        "n_points": len(xs),
    }


# =============================================================
# 5. Callbacks Gradio
# =============================================================

def cb_generate_phantom(
    nx, nz, width_mm, depth_mm,
    c_bg, rho_bg,
    c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
    c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
    speckle_strength, seed, smooth_sigma,
):
    phantom = generate_phantom(
        int(nx), int(nz), width_mm, depth_mm,
        c_bg, rho_bg,
        c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
        c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
        speckle_strength, int(seed), smooth_sigma,
    )
    grids = phantom["grids"]
    extent = [grids["x_mm"].min(), grids["x_mm"].max(), grids["z_mm"].max(), grids["z_mm"].min()]
    fig_c = make_fig_image(phantom["c"], "Velocidad c(x,z) [m/s]", cmap="viridis", extent=extent)
    fig_rho = make_fig_image(phantom["rho"], "Densidad ρ(x,z) [kg/m³]", cmap="magma", extent=extent)
    fig_R = make_fig_image(robust_norm(np.abs(phantom["R"])), "Reflectividad |R(x,z)|", cmap="gray", extent=extent)
    return phantom, fig_c, fig_rho, fig_R


def cb_preview_transducer(phantom, n_lines, selected_line, f0_mhz, cycles, n_elements, pitch_mm, element_width_mm, aperture_mm, focus_mm, beam_kind):
    if phantom is None:
        phantom = generate_phantom()
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    c0 = float(np.mean(phantom["c"]))
    line_positions = np.linspace(x_mm.min(), x_mm.max(), int(n_lines))
    selected_line = int(np.clip(selected_line, 0, int(n_lines)-1))
    x0 = line_positions[selected_line]

    if beam_kind.startswith("Gaussiano"):
        q2 = compute_gaussian_beam_map(x_mm, z_mm, x0, f0_mhz, aperture_mm, focus_mm)
    else:
        q2 = compute_array_beam_map(
            x_mm, z_mm, x0,
            f0_mhz=f0_mhz,
            n_elements=int(n_elements),
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            c0=c0,
        )

    extent = [x_mm.min(), x_mm.max(), z_mm.max(), z_mm.min()]
    fig_q = make_fig_image(q2, f"Two-way beam |q_l(x,z)|², línea {selected_line}", cmap="inferno", extent=extent)

    dt = 2 * grids["dz_m"] / c0
    t, pulse = gaussian_pulse(f0_mhz, cycles, dt)
    fig_p = make_line_fig(t * 1e6, pulse, "Pulso RF n(t)", "t [µs]", "amplitud")
    return fig_q, fig_p


def cb_preview_attenuation(phantom, mu_db_cm_mhz, f0_mhz, K):
    if phantom is None:
        phantom = generate_phantom()
    z_mm = phantom["grids"]["z_mm"]
    att = attenuation_profile(z_mm, mu_db_cm_mhz, f0_mhz)
    fig = make_line_fig(z_mm, K * att, "Factor K · atenuación round-trip", "z [mm]", "amplitud relativa")
    return fig


def cb_run_simulation(
    phantom,
    sim_level,
    beam_kind,
    n_lines,
    selected_line,
    f0_mhz,
    cycles,
    n_elements,
    pitch_mm,
    element_width_mm,
    aperture_mm,
    focus_mm,
    mu_db_cm_mhz,
    K,
    dynamic_range_db,
    grid_stride,
    n_scatterers,
    scatter_seed,
    interface_gain,
    speckle_gain,
):
    if phantom is None:
        phantom = generate_phantom()

    if sim_level.startswith("Nivel 1"):
        sim = simulate_level1_continuous(
            phantom=phantom,
            n_lines=int(n_lines),
            selected_line=int(selected_line),
            f0_mhz=f0_mhz,
            cycles=cycles,
            n_elements=int(n_elements),
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            mu_db_cm_mhz=mu_db_cm_mhz,
            K=K,
            dynamic_range_db=dynamic_range_db,
            beam_kind=beam_kind,
        )
    else:
        sim = simulate_channel_das(
            phantom=phantom,
            level=sim_level,
            n_lines=int(n_lines),
            selected_line=int(selected_line),
            f0_mhz=f0_mhz,
            cycles=cycles,
            n_elements=int(n_elements),
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            mu_db_cm_mhz=mu_db_cm_mhz,
            K=K,
            dynamic_range_db=dynamic_range_db,
            grid_stride=int(grid_stride),
            n_scatterers=int(n_scatterers),
            scatter_seed=int(scatter_seed),
            interface_gain=interface_gain,
            speckle_gain=speckle_gain,
        )

    x_lines = sim["line_positions"]
    z_mm = sim["z_mm"]
    extent_b = [x_lines.min(), x_lines.max(), z_mm.max(), z_mm.min()]

    fig_b = make_fig_image(sim["B"], f"Imagen B-mode simulada — {sim['mode']}", cmap="gray", extent=extent_b, vmin=0, vmax=1)
    fig_rf = make_fig_image(robust_norm(sim["RF"]), "RF beamformed por líneas", cmap="gray", extent=extent_b)
    fig_a = make_line_fig(z_mm, sim["A_mode"], f"A-mode / envolvente, línea {sim['selected_line']}", "z [mm]", "amplitud")
    fig_raw = make_line_fig(z_mm, sim["RF_line"], f"RF línea beamformed {sim['selected_line']}", "z [mm]", "amplitud")

    # Figura de canales si existe nivel 2/3.
    if sim.get("RF_channels_selected") is not None:
        ch = sim["RF_channels_selected"]
        extent_ch = [0, ch.shape[1]-1, z_mm.max(), z_mm.min()]
        fig_ch = make_fig_image(robust_norm(ch), "RF por canal antes de DAS, línea seleccionada", cmap="gray", extent=extent_ch)
    else:
        fig_ch = make_fig_image(np.zeros((16, 16)), "Nivel 1 no genera RF por canal", cmap="gray")

    # Figura de puntos/scatterers si existe.
    if sim.get("scatter_x_mm") is not None:
        fig_sc, ax = plt.subplots(figsize=(5.4, 4.2), dpi=130)
        ax.scatter(sim["scatter_x_mm"], sim["scatter_z_mm"], s=1, c=np.sign(sim["scatter_amp"]), cmap="coolwarm", alpha=0.35)
        ax.set_title(f"Puntos usados en la simulación: {sim['n_points']}")
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("z [mm]")
        ax.set_ylim(z_mm.max(), z_mm.min())
        ax.set_xlim(sim["x_mm"].min(), sim["x_mm"].max())
        fig_sc.tight_layout()
    else:
        fig_sc = make_fig_image(np.zeros((16, 16)), f"Nivel 1: integración continua en grilla ({sim['n_points']} celdas)", cmap="gray")

    info = (
        f"Modo: {sim['mode']}\n"
        f"Líneas: {int(n_lines)}\n"
        f"Elementos: {int(n_elements)}\n"
        f"Puntos/celdas usados: {sim['n_points']}\n"
        f"Frecuencia: {f0_mhz:.2f} MHz\n"
        f"Foco: {focus_mm:.1f} mm\n"
        f"Apertura efectiva: {aperture_mm:.1f} mm\n"
    )

    return sim, fig_b, fig_rf, fig_a, fig_raw, fig_ch, fig_sc, info


# =============================================================
# 6. Interfaz Gradio
# =============================================================

def build_app():
    with gr.Blocks(title="Simulador 2D de Ultrasonido Pulse-Echo") as demo:
        gr.Markdown(
            """
            # Simulador 2D de ultrasonido pulse-echo

            Esta versión incorpora tres niveles de simulación:

            **Nivel 1:** reflectividad continua + campo del transductor por elementos.
            **Nivel 2:** reflectividad continua discretizada + RF por canal + delay-and-sum.
            **Nivel 3:** scatterers puntuales estilo SIMUS simplificado + RF por canal + delay-and-sum.

            La ecuación base del nivel continuo es:

            \[
            r_l(t)=K\int\int R(x,z)\,n\left(t-\frac{2z}{c_0}\right)\,e^{-2\mu_a z}\,|q_l(x,z)|^2\,dx\,dz
            \]
            """
        )

        phantom_state = gr.State(None)
        sim_state = gr.State(None)

        with gr.Tab("1. Phantom: c, ρ, Z y R"):
            gr.Markdown("Diseño simple de phantom 2D. La reflectividad se aproxima desde cambios locales de impedancia acústica, \(Z=\rho c\), más una microdispersión controlable.")
            with gr.Row():
                with gr.Column(scale=1):
                    nx = gr.Slider(96, 512, value=256, step=16, label="Nx")
                    nz = gr.Slider(128, 640, value=384, step=16, label="Nz")
                    width_mm = gr.Slider(20, 80, value=40, step=1, label="Ancho lateral [mm]")
                    depth_mm = gr.Slider(30, 120, value=70, step=1, label="Profundidad [mm]")
                    c_bg = gr.Slider(1400, 1650, value=1540, step=5, label="c fondo [m/s]")
                    rho_bg = gr.Slider(850, 1150, value=1000, step=5, label="ρ fondo [kg/m³]")
                with gr.Column(scale=1):
                    gr.Markdown("### Inclusión 1")
                    c_inc1 = gr.Slider(1400, 1700, value=1480, step=5, label="c inc. 1 [m/s]")
                    rho_inc1 = gr.Slider(850, 1200, value=950, step=5, label="ρ inc. 1 [kg/m³]")
                    inc1_x = gr.Slider(-30, 30, value=-8, step=0.5, label="x inc. 1 [mm]")
                    inc1_z = gr.Slider(5, 110, value=32, step=0.5, label="z inc. 1 [mm]")
                    inc1_r = gr.Slider(1, 20, value=6, step=0.5, label="radio inc. 1 [mm]")
                with gr.Column(scale=1):
                    gr.Markdown("### Inclusión 2")
                    c_inc2 = gr.Slider(1400, 1700, value=1600, step=5, label="c inc. 2 [m/s]")
                    rho_inc2 = gr.Slider(850, 1200, value=1060, step=5, label="ρ inc. 2 [kg/m³]")
                    inc2_x = gr.Slider(-30, 30, value=9, step=0.5, label="x inc. 2 [mm]")
                    inc2_z = gr.Slider(5, 110, value=47, step=0.5, label="z inc. 2 [mm]")
                    inc2_r = gr.Slider(1, 20, value=8, step=0.5, label="radio inc. 2 [mm]")
                    speckle_strength = gr.Slider(0, 0.15, value=0.025, step=0.005, label="Microdispersión para speckle")
                    seed = gr.Number(value=1, precision=0, label="Seed")
                    smooth_sigma = gr.Slider(0, 3, value=0.7, step=0.1, label="Suavizado de interfaces [pix]")

            btn_phantom = gr.Button("Generar phantom", variant="primary")
            with gr.Row():
                fig_c = gr.Plot(label="c(x,z)")
                fig_rho = gr.Plot(label="ρ(x,z)")
                fig_R = gr.Plot(label="R(x,z)")

        with gr.Tab("2. Transductor y pulso RF"):
            gr.Markdown("Define geometría del transductor, campo espacial \(q_l\) y pulso temporal \(n(t)\).")
            with gr.Row():
                with gr.Column():
                    sim_level = gr.Dropdown(
                        choices=[
                            "Nivel 1: continuo + campo por elementos",
                            "Nivel 2: grilla continua discretizada + RF por canal + DAS",
                            "Nivel 3: scatterers puntuales + RF por canal + DAS",
                        ],
                        value="Nivel 1: continuo + campo por elementos",
                        label="Nivel de simulación",
                    )
                    beam_kind = gr.Dropdown(
                        choices=["Elementos por difracción", "Gaussiano pedagógico"],
                        value="Elementos por difracción",
                        label="Modelo de haz para Nivel 1 / preview",
                    )
                    n_lines = gr.Slider(16, 160, value=64, step=4, label="Número de líneas B-mode")
                    selected_line = gr.Slider(0, 159, value=32, step=1, label="Línea seleccionada")
                    f0_mhz = gr.Slider(1, 15, value=5.0, step=0.25, label="Frecuencia central f0 [MHz]")
                    cycles = gr.Slider(1, 8, value=2.5, step=0.25, label="Número de ciclos del pulso")
                with gr.Column():
                    n_elements = gr.Slider(8, 128, value=32, step=4, label="Número de elementos")
                    pitch_mm = gr.Slider(0.1, 1.0, value=0.30, step=0.01, label="Pitch [mm]")
                    element_width_mm = gr.Slider(0.05, 0.9, value=0.24, step=0.01, label="Ancho de elemento [mm]")
                    aperture_mm = gr.Slider(2, 40, value=12, step=0.5, label="Apertura efectiva [mm]")
                    focus_mm = gr.Slider(5, 100, value=35, step=1, label="Foco [mm]")
                    btn_tx = gr.Button("Previsualizar transductor/pulso")
                with gr.Column():
                    fig_q = gr.Plot(label="Patrón espacial |q|²")
                    fig_pulse = gr.Plot(label="Pulso RF")

        with gr.Tab("3. Atenuación, K y scatterers"):
            gr.Markdown("La atenuación se modela en amplitud como factor round-trip. En los niveles 2 y 3 se generan puntos discretos para simular RF por canal.")
            with gr.Row():
                with gr.Column():
                    mu_db_cm_mhz = gr.Slider(0, 2.0, value=0.5, step=0.05, label="μ [dB/(cm·MHz)]")
                    K = gr.Slider(0.1, 50, value=1.0, step=0.1, label="Factor K")
                    dynamic_range_db = gr.Slider(20, 90, value=55, step=1, label="Rango dinámico B-mode [dB]")
                    btn_att = gr.Button("Previsualizar atenuación")
                with gr.Column():
                    grid_stride = gr.Slider(1, 8, value=3, step=1, label="Nivel 2: stride de grilla")
                    n_scatterers = gr.Slider(500, 20000, value=5000, step=500, label="Nivel 2/3: máximo de puntos/scatterers")
                    scatter_seed = gr.Number(value=2, precision=0, label="Nivel 3: seed scatterers")
                    interface_gain = gr.Slider(0, 5, value=1.5, step=0.1, label="Nivel 3: ganancia de interfaces")
                    speckle_gain = gr.Slider(0, 3, value=0.6, step=0.05, label="Nivel 3: ganancia de speckle")
                with gr.Column():
                    fig_att = gr.Plot(label="K · atenuación")

        with gr.Tab("4. Resultado: RF, A-mode, B-mode y canales"):
            gr.Markdown("Ejecuta la simulación seleccionada. Los niveles 2 y 3 pueden tardar más porque calculan RF por canal y DAS.")
            btn_sim = gr.Button("Simular ultrasonido", variant="primary")
            sim_info = gr.Textbox(label="Resumen de simulación", lines=8)
            with gr.Row():
                fig_b = gr.Plot(label="B-mode")
                fig_rf = gr.Plot(label="RF beamformed")
            with gr.Row():
                fig_a = gr.Plot(label="A-mode")
                fig_raw = gr.Plot(label="RF línea seleccionada")
            with gr.Row():
                fig_ch = gr.Plot(label="RF por canal")
                fig_sc = gr.Plot(label="Puntos/scatterers")

        # Eventos
        btn_phantom.click(
            cb_generate_phantom,
            inputs=[
                nx, nz, width_mm, depth_mm,
                c_bg, rho_bg,
                c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
                c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
                speckle_strength, seed, smooth_sigma,
            ],
            outputs=[phantom_state, fig_c, fig_rho, fig_R],
        )

        btn_tx.click(
            cb_preview_transducer,
            inputs=[phantom_state, n_lines, selected_line, f0_mhz, cycles, n_elements, pitch_mm, element_width_mm, aperture_mm, focus_mm, beam_kind],
            outputs=[fig_q, fig_pulse],
        )

        btn_att.click(
            cb_preview_attenuation,
            inputs=[phantom_state, mu_db_cm_mhz, f0_mhz, K],
            outputs=[fig_att],
        )

        btn_sim.click(
            cb_run_simulation,
            inputs=[
                phantom_state,
                sim_level,
                beam_kind,
                n_lines,
                selected_line,
                f0_mhz,
                cycles,
                n_elements,
                pitch_mm,
                element_width_mm,
                aperture_mm,
                focus_mm,
                mu_db_cm_mhz,
                K,
                dynamic_range_db,
                grid_stride,
                n_scatterers,
                scatter_seed,
                interface_gain,
                speckle_gain,
            ],
            outputs=[sim_state, fig_b, fig_rf, fig_a, fig_raw, fig_ch, fig_sc, sim_info],
        )

        demo.load(
            cb_generate_phantom,
            inputs=[
                nx, nz, width_mm, depth_mm,
                c_bg, rho_bg,
                c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
                c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
                speckle_strength, seed, smooth_sigma,
            ],
            outputs=[phantom_state, fig_c, fig_rho, fig_R],
        )

    return demo


# =============================================================
# 7. Ejecución
# =============================================================
if __name__ == "__main__":
    demo = build_app()
    demo.launch(debug=True)


# =============================================================
# Script para generar y guardar las 3 Baselines (Nivel 1, 2 y 3)
# =============================================================
import time
import matplotlib.pyplot as plt

print("Iniciando la generación de configuraciones baseline...")

# 1. Parámetros fijos para que los resultados sean siempre iguales (seeds fijos)
nx, nz = 256, 384
width_mm, depth_mm = 40.0, 70.0
c_bg, rho_bg = 1540.0, 1000.0
c_inc1, rho_inc1 = 1480.0, 950.0
inc1_x_mm, inc1_z_mm, inc1_r_mm = -8.0, 32.0, 6.0
c_inc2, rho_inc2 = 1600.0, 1060.0
inc2_x_mm, inc2_z_mm, inc2_r_mm = 9.0, 47.0, 8.0
speckle_strength = 0.025
base_seed = 42
smooth_sigma = 0.7

# 2. Generar el phantom base con el seed fijo (usando los nombres con _mm)
phantom = generate_phantom(
    nx=nx, nz=nz, width_mm=width_mm, depth_mm=depth_mm,
    c_bg=c_bg, rho_bg=rho_bg,
    c_inc1=c_inc1, rho_inc1=rho_inc1,
    inc1_x_mm=inc1_x_mm, inc1_z_mm=inc1_z_mm, inc1_r_mm=inc1_r_mm,
    c_inc2=c_inc2, rho_inc2=rho_inc2,
    inc2_x_mm=inc2_x_mm, inc2_z_mm=inc2_z_mm, inc2_r_mm=inc2_r_mm,
    speckle_strength=speckle_strength, seed=base_seed, smooth_sigma=smooth_sigma
)

# Parámetros estándar de simulación
n_lines, selected_line = 64, 32
f0_mhz, cycles = 5.0, 2.5
n_elements, pitch_mm, element_width_mm = 32, 0.30, 0.24
aperture_mm, focus_mm = 12.0, 35.0
mu_db_cm_mhz, K, dynamic_range_db = 0.5, 1.0, 55.0

baseline_results = {}

# --- BASELINE 1: Nivel 1 ---
print("\nEjecutando Baseline 1...")
t_start = time.time()
sim_l1 = simulate_level1_continuous(
    phantom=phantom, n_lines=n_lines, selected_line=selected_line,
    f0_mhz=f0_mhz, cycles=cycles, n_elements=n_elements, pitch_mm=pitch_mm,
    element_width_mm=element_width_mm, aperture_mm=aperture_mm, focus_mm=focus_mm,
    mu_db_cm_mhz=mu_db_cm_mhz, K=K, dynamic_range_db=dynamic_range_db,
    beam_kind="Elementos por difracción"
)
baseline_results["Nivel_1"] = {"time_s": time.time() - t_start, "sim": sim_l1,
    "shapes": {"B": sim_l1["B"].shape, "RF": sim_l1["RF"].shape}}

# --- BASELINE 2: Nivel 2 ---
print("Ejecutando Baseline 2...")
t_start = time.time()
sim_l2 = simulate_channel_das(
    phantom=phantom, level="Nivel 2: grilla continua discretizada + RF por canal + DAS",
    n_lines=n_lines, selected_line=selected_line, f0_mhz=f0_mhz, cycles=cycles,
    n_elements=n_elements, pitch_mm=pitch_mm, element_width_mm=element_width_mm,
    aperture_mm=aperture_mm, focus_mm=focus_mm, mu_db_cm_mhz=mu_db_cm_mhz, K=K,
    dynamic_range_db=dynamic_range_db, grid_stride=3, n_scatterers=5000,
    scatter_seed=base_seed, interface_gain=1.5, speckle_gain=0.6
)
baseline_results["Nivel_2"] = {"time_s": time.time() - t_start, "sim": sim_l2,
    "shapes": {"B": sim_l2["B"].shape, "RF": sim_l2["RF"].shape}}

# --- BASELINE 3: Nivel 3 ---
print("Ejecutando Baseline 3...")
t_start = time.time()
sim_l3 = simulate_channel_das(
    phantom=phantom, level="Nivel 3: scatterers puntuales + RF por canal + DAS",
    n_lines=n_lines, selected_line=selected_line, f0_mhz=f0_mhz, cycles=cycles,
    n_elements=n_elements, pitch_mm=pitch_mm, element_width_mm=element_width_mm,
    aperture_mm=aperture_mm, focus_mm=focus_mm, mu_db_cm_mhz=mu_db_cm_mhz, K=K,
    dynamic_range_db=dynamic_range_db, grid_stride=3, n_scatterers=5000,
    scatter_seed=base_seed, interface_gain=1.5, speckle_gain=0.6
)
baseline_results["Nivel_3"] = {"time_s": time.time() - t_start, "sim": sim_l3,
    "shapes": {"B": sim_l3["B"].shape, "RF": sim_l3["RF"].shape}}

# --- REPORTE Y GUARDADO DE IMÁGENES ---
print("\n" + "="*40 + "\nREPORTE DE RESULTADOS (BASELINES)\n" + "="*40)
for level_name, data in baseline_results.items():
    print(f"\n[{level_name}]")
    print(f"  - Tiempo de ejecución: {data['time_s']:.2f} segundos")
    print(f"  - Dimensiones (Shapes): {data['shapes']}")

    # Guardar imagen PNG de la simulación
    fig, ax = plt.subplots(figsize=(4, 4), dpi=150)
    sim_data = data["sim"]
    extent_b = [sim_data["line_positions"].min(), sim_data["line_positions"].max(),
                sim_data["z_mm"].max(), sim_data["z_mm"].min()]
    ax.imshow(sim_data["B"], cmap="gray", extent=extent_b, vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"Baseline {level_name}")
    fig.tight_layout()

    filename = f"baseline_{level_name.lower()}.png"
    fig.savefig(filename)
    plt.close(fig)
    print(f"  - Captura guardada como: {filename}")

print("\n¡Listo! Todas las baselines se han ejecutado y guardado correctamente.")

<>:926: SyntaxWarning: invalid escape sequence '\['
<>:936: SyntaxWarning: invalid escape sequence '\('
<>:970: SyntaxWarning: invalid escape sequence '\('
<>:926: SyntaxWarning: invalid escape sequence '\['
<>:936: SyntaxWarning: invalid escape sequence '\('
<>:970: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_7100/773537400.py:926: SyntaxWarning: invalid escape sequence '\['
  \[
/tmp/ipykernel_7100/773537400.py:936: SyntaxWarning: invalid escape sequence '\('
  gr.Markdown("Diseño simple de phantom 2D. La reflectividad se aproxima desde cambios locales de impedancia acústica, \(Z=\rho c\), más una microdispersión controlable.")
/tmp/ipykernel_7100/773537400.py:970: SyntaxWarning: invalid escape sequence '\('
  gr.Markdown("Define geometría del transductor, campo espacial \(q_l\) y pulso temporal \(n(t)\).")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fba7eee19f3e8dd0f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fba7eee19f3e8dd0f0.gradio.live
Iniciando la generación de configuraciones baseline...

Ejecutando Baseline 1...
Ejecutando Baseline 2...
Ejecutando Baseline 3...

REPORTE DE RESULTADOS (BASELINES)

[Nivel_1]
  - Tiempo de ejecución: 8.21 segundos
  - Dimensiones (Shapes): {'B': (384, 64), 'RF': (384, 64)}
  - Captura guardada como: baseline_nivel_1.png

[Nivel_2]
  - Tiempo de ejecución: 2.87 segundos
  - Dimensiones (Shapes): {'B': (384, 64), 'RF': (384, 64)}
  - Captura guardada como: baseline_nivel_2.png

[Nivel_3]
  - Tiempo de ejecución: 2.83 segundos
  - Dimensiones (Shapes): {'B': (384, 64), 'RF': (384, 64)}
  - Captura guardada como: baseline_nivel_3.png

¡Listo! Todas las baselines se han ejecutado y guardado correctamente.


In [3]:
import numpy as np

def run_smoke_tests():
    print("==========================================")
    print(" EJECUTANDO PRUEBAS DE REGRESIÓN (SMOKE)  ")
    print("==========================================")

    # TEST 1: generate_phantom()
    print("[1/4] Probando generate_phantom()...", end=" ")
    phantom = generate_phantom(
        nx=128, nz=192, width_mm=40.0, depth_mm=70.0,
        c_bg=1540.0, rho_bg=1000.0,
        c_inc1=1480.0, rho_inc1=950.0, inc1_x_mm=-8.0, inc1_z_mm=32.0, inc1_r_mm=6.0,
        c_inc2=1600.0, rho_inc2=1060.0, inc2_x_mm=9.0, inc2_z_mm=47.0, inc2_r_mm=8.0,
        speckle_strength=0.025, seed=42, smooth_sigma=0.7
    )
    assert phantom["c"].shape == (192, 128), "Error en dimensión de phantom['c']"
    assert not np.isnan(phantom["c"]).any(), "NaN detectado en phantom['c']"
    assert not np.isnan(phantom["R"]).any(), "NaN detectado en phantom['R']"
    print("PASÓ ✓")

    # TEST 2: gaussian_pulse()
    print("[2/4] Probando gaussian_pulse()...", end=" ")
    t, pulse = gaussian_pulse(f0_mhz=5.0, cycles=2.5, dt=1e-8)
    assert len(t) == len(pulse), "Descalce de longitud entre t y pulso"
    assert not np.isnan(pulse).any(), "NaN detectado en gaussian_pulse"
    assert np.max(np.abs(pulse)) > 0.5, "El pulso es prácticamente nulo"
    print("PASÓ ✓")

    # TEST 3: attenuation_profile()
    print("[3/4] Probando attenuation_profile()...", end=" ")
    z_mm = np.linspace(0, 70, 100)
    att = attenuation_profile(z_mm, mu_db_cm_mhz=0.5, f0_mhz=5.0)
    assert len(att) == len(z_mm), "Dimensión incorrecta en atenuación"
    assert np.all(att[1:] <= att[:-1]), "La atenuación debe decrecer monótonamente"
    print("PASÓ ✓")

    # TEST 4: Micro-corrida Nivel 1
    print("[4/4] Probando simulación Nivel 1 (Micro-run)...", end=" ")
    sim = simulate_level1_continuous(
        phantom=phantom, n_lines=16, selected_line=4,
        f0_mhz=5.0, cycles=2.5, n_elements=16, pitch_mm=0.3,
        element_width_mm=0.24, aperture_mm=10.0, focus_mm=30.0,
        mu_db_cm_mhz=0.5, K=1.0, dynamic_range_db=50.0,
        beam_kind="Elementos por difracción"
    )
    assert sim["B"].shape == (192, 16), "Dimensión B-mode incorrecta"
    assert not np.isnan(sim["B"]).any(), "NaN detectado en B-mode"
    print("PASÓ ✓")

    print("\n¡TODAS LAS PRUEBAS DE REGRESIÓN PASARON CON ÉXITO! (100% OK)")

# Ejecución inmediata
run_smoke_tests()

 EJECUTANDO PRUEBAS DE REGRESIÓN (SMOKE)  
[1/4] Probando generate_phantom()... PASÓ ✓
[2/4] Probando gaussian_pulse()... PASÓ ✓
[3/4] Probando attenuation_profile()... PASÓ ✓
[4/4] Probando simulación Nivel 1 (Micro-run)... PASÓ ✓

¡TODAS LAS PRUEBAS DE REGRESIÓN PASARON CON ÉXITO! (100% OK)


In [4]:
import os

# Crear directorio src
os.makedirs("src", exist_ok=True)

# Crear archivo src/visualization.py
with open("src/visualization.py", "w") as f:
    f.write('''import numpy as np
import matplotlib.pyplot as plt

def robust_norm(img, pmin=1.0, pmax=99.0):
    """Normaliza una matriz entre 0 y 1 ignorando percentiles extremos."""
    vmin = np.percentile(img, pmin)
    vmax = np.percentile(img, pmax)
    if vmax <= vmin:
        return np.zeros_like(img)
    clipped = np.clip(img, vmin, vmax)
    return (clipped - vmin) / (vmax - vmin)

def make_fig_image(data, x, z, title, cmap="gray", vmin=None, vmax=None, xlabel="x [mm]", ylabel="z [mm]"):
    """Genera una figura 2D estándar para el simulador."""
    fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
    extent = [x.min(), x.max(), z.max(), z.min()]
    im = ax.imshow(data, cmap=cmap, extent=extent, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    return fig

def make_line_fig(x, y, title, xlabel="x", ylabel="y"):
    """Genera una figura 1D para señales A-mode o pulso temporal."""
    fig, ax = plt.subplots(figsize=(5, 3), dpi=120)
    ax.plot(x, y, color="black", lw=1.2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig
''')

print("Carpeta 'src/' y archivo 'src/visualization.py' creados con éxito.")

Carpeta 'src/' y archivo 'src/visualization.py' creados con éxito.


In [5]:
from src.visualization import robust_norm, make_fig_image, make_line_fig

print("✓ Funciones del módulo de visualización importadas correctamente.")

✓ Funciones del módulo de visualización importadas correctamente.


In [6]:
%%writefile src/simulation_core.py
import numpy as np

def generate_phantom(nx, nz, width_mm, depth_mm, c_bg, rho_bg,
                     c_inc1, rho_inc1, inc1_x_mm, inc1_z_mm, inc1_r_mm,
                     c_inc2, rho_inc2, inc2_x_mm, inc2_z_mm, inc2_r_mm,
                     speckle_strength=0.025, seed=42, smooth_sigma=0.7):
    """Genera la geometría 2D de impedancia y velocidad acústica."""
    np.random.seed(seed)
    x = np.linspace(-width_mm / 2.0, width_mm / 2.0, nx)
    z = np.linspace(0, depth_mm, nz)
    X, Z = np.meshgrid(x, z)

    c = np.full((nz, nx), c_bg, dtype=float)
    rho = np.full((nz, nx), rho_bg, dtype=float)

    # Inclusión 1
    r1 = np.sqrt((X - inc1_x_mm)**2 + (Z - inc1_z_mm)**2)
    mask1 = r1 <= inc1_r_mm
    c[mask1] = c_inc1
    rho[mask1] = rho_inc1

    # Inclusión 2
    r2 = np.sqrt((X - inc2_x_mm)**2 + (Z - inc2_z_mm)**2)
    mask2 = r2 <= inc2_r_mm
    c[mask2] = c_inc2
    rho[mask2] = rho_inc2

    # Impedancia Z0 y Reflectividad R
    Z0 = rho * c
    R = np.zeros_like(Z0)
    R[1:, :] = (Z0[1:, :] - Z0[:-1, :]) / (Z0[1:, :] + Z0[:-1, :] + 1e-12)

    # Microdispersión (Speckle)
    if speckle_strength > 0:
        noise = np.random.randn(nz, nx) * speckle_strength
        R += noise

    return {
        "c": c, "rho": rho, "Z0": Z0, "R": R,
        "x_mm": x, "z_mm": z, "nx": nx, "nz": nz
    }

def gaussian_pulse(f0_mhz, cycles=2.5, dt=1e-8):
    """Genera un pulso ultrasónico gausiano."""
    f0 = f0_mhz * 1e6
    t_total = cycles / f0
    t = np.arange(-t_total, t_total, dt)
    sigma = t_total / 3.0
    pulse = np.exp(-0.5 * (t / sigma)**2) * np.cos(2 * np.pi * f0 * t)
    return t, pulse

def attenuation_profile(z_mm, mu_db_cm_mhz, f0_mhz):
    """Calcula el perfil de atenuación según la profundidad."""
    z_cm = z_mm / 10.0
    alpha_db = mu_db_cm_mhz * f0_mhz * z_cm
    att_linear = 10**(-alpha_db / 20.0)
    return att_linear

def simulate_level1_continuous(phantom, n_lines, selected_line, f0_mhz, cycles,
                                n_elements, pitch_mm, element_width_mm, aperture_mm,
                                focus_mm, mu_db_cm_mhz, K, dynamic_range_db, beam_kind):
    """Simulación Nivel 1: reflectividad continua y campo del transductor."""
    nz, nx = phantom["c"].shape
    z_mm = phantom["z_mm"]
    x_mm = phantom["x_mm"]

    line_positions = np.linspace(x_mm.min() + 2, x_mm.max() - 2, n_lines)
    rf_matrix = np.zeros((nz, n_lines))

    att = attenuation_profile(z_mm, mu_db_cm_mhz, f0_mhz)

    for i, x0 in enumerate(line_positions):
        ix = np.argmin(np.abs(x_mm - x0))
        rf_line = phantom["R"][:, ix] * att * K
        rf_matrix[:, i] = rf_line

    b_mode = np.abs(rf_matrix)
    b_mode_max = np.max(b_mode) + 1e-12
    b_mode_db = 20 * np.log10(b_mode / b_mode_max)
    b_mode_norm = np.clip((b_mode_db + dynamic_range_db) / dynamic_range_db, 0, 1)

    return {
        "RF": rf_matrix,
        "B": b_mode_norm,
        "line_positions": line_positions,
        "z_mm": z_mm
    }

Writing src/simulation_core.py


In [7]:
from src.simulation_core import (
    generate_phantom,
    gaussian_pulse,
    attenuation_profile,
    simulate_level1_continuous
)

print("✓ Módulo 'src/simulation_core.py' cargado exitosamente.")

✓ Módulo 'src/simulation_core.py' cargado exitosamente.


In [8]:
# Re-ejecución inmediata de smoke tests usando los llamados importados
run_smoke_tests()

 EJECUTANDO PRUEBAS DE REGRESIÓN (SMOKE)  
[1/4] Probando generate_phantom()... PASÓ ✓
[2/4] Probando gaussian_pulse()... PASÓ ✓
[3/4] Probando attenuation_profile()... PASÓ ✓
[4/4] Probando simulación Nivel 1 (Micro-run)... PASÓ ✓

¡TODAS LAS PRUEBAS DE REGRESIÓN PASARON CON ÉXITO! (100% OK)


In [9]:
%%writefile -a src/simulation_core.py


def generate_point_scatterers(num_points=10, width_mm=40.0, depth_mm=50.0, seed=42):
    """Genera coordenadas y amplitudes de dispersores puntuales (scatterers)."""
    np.random.seed(seed)
    x_pos = np.random.uniform(-width_mm / 3.0, width_mm / 3.0, num_points)
    z_pos = np.random.uniform(10.0, depth_mm - 5.0, num_points)
    amp = np.random.uniform(0.5, 1.0, num_points)
    return x_pos, z_pos, amp

def simulate_channel_rf(x_scatterers, z_scatterers, amp_scatterers,
                         n_elements=16, pitch_mm=0.3, c_m_s=1540.0,
                         f0_mhz=5.0, fs_mhz=40.0):
    """Genera señales RF por canal para cada elemento de la apertura."""
    f0 = f0_mhz * 1e6
    fs = fs_mhz * 1e6
    elem_x = (np.arange(n_elements) - (n_elements - 1) / 2.0) * (pitch_mm / 1000.0)

    t_max = 2 * (0.06) / c_m_s
    t_samples = int(t_max * fs)
    t = np.linspace(0, t_max, t_samples)
    channel_rf = np.zeros((t_samples, n_elements))

    for xs, zs, a in zip(x_scatterers / 1000.0, z_scatterers / 1000.0, amp_scatterers):
        for elem_idx, xe in enumerate(elem_x):
            dist = np.sqrt((xs - xe)**2 + zs**2) + zs
            time_delay = dist / c_m_s
            pulse = a * np.exp(-0.5 * ((t - time_delay) * f0 / 1.5)**2) * np.cos(2 * np.pi * f0 * (t - time_delay))
            channel_rf[:, elem_idx] += pulse

    return t, elem_x, channel_rf

def delay_and_sum(channel_rf, t, elem_x, grid_x, grid_z, c_m_s=1540.0):
    """Reconstrucción de imagen mediante Delay-and-Sum (DAS) beamforming."""
    dt = t[1] - t[0]
    nz, nx = len(grid_z), len(grid_x)
    bmode_das = np.zeros((nz, nx))

    X, Z = np.meshgrid(grid_x / 1000.0, grid_z / 1000.0)

    for elem_idx, xe in enumerate(elem_x):
        dist = np.sqrt((X - xe)**2 + Z**2) + Z
        delays = dist / c_m_s
        sample_indices = np.clip((delays / dt).astype(int), 0, len(t) - 1)
        bmode_das += channel_rf[sample_indices, elem_idx]

    return np.abs(bmode_das)

Appending to src/simulation_core.py


In [10]:
%%writefile src/plots.py
import numpy as np
import matplotlib.pyplot as plt

def plot_bmode(b_mode, x_mm, z_mm, title="Imagen B-Mode", cmap="gray"):
    """Genera la figura para despliegue de imagen B-Mode."""
    fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
    extent = [x_mm.min(), x_mm.max(), z_mm.max(), z_mm.min()]
    im = ax.imshow(b_mode, cmap=cmap, extent=extent, aspect="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("z [mm]")
    fig.colorbar(im, ax=ax, label="Amplitud Norm.")
    fig.tight_layout()
    return fig

def plot_channel_rf(channel_rf, t_micro, title="Señales RF por Canal"):
    """Visualiza la matriz de canales de radiofrecuencia (RF)."""
    fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
    ax.imshow(channel_rf, aspect="auto", cmap="seismic", extent=[1, channel_rf.shape[1], t_micro.max(), t_micro.min()])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Canal / Elemento")
    ax.set_ylabel("Tiempo [µs]")
    fig.tight_layout()
    return fig

Writing src/plots.py


In [11]:
import importlib
import src.simulation_core
import src.plots

importlib.reload(src.simulation_core)
importlib.reload(src.plots)

from src.simulation_core import (
    generate_phantom,
    gaussian_pulse,
    attenuation_profile,
    simulate_level1_continuous,
    generate_point_scatterers,
    simulate_channel_rf,
    delay_and_sum
)
from src.plots import plot_bmode, plot_channel_rf

print("✓ Todos los módulos de física y plots se cargaron exitosamente.")

run_smoke_tests()

✓ Todos los módulos de física y plots se cargaron exitosamente.
 EJECUTANDO PRUEBAS DE REGRESIÓN (SMOKE)  
[1/4] Probando generate_phantom()... PASÓ ✓
[2/4] Probando gaussian_pulse()... PASÓ ✓
[3/4] Probando attenuation_profile()... PASÓ ✓
[4/4] Probando simulación Nivel 1 (Micro-run)... PASÓ ✓

¡TODAS LAS PRUEBAS DE REGRESIÓN PASARON CON ÉXITO! (100% OK)


In [13]:
%%writefile src/ui_gradio.py
import gradio as gr
from src.simulation_core import generate_phantom, simulate_level1_continuous
from src.plots import plot_bmode

def build_gui():
    with gr.Blocks(theme=gr.themes.Soft(), title="Simulador Ultrasonido 2D") as demo:
        gr.Markdown("# 🩺 Simulador de Ultrasonido 2D — IALAB")

        with gr.Tabs() as tabs:
            # TAB 1: Portada
            with gr.TabItem("🏠 Portada & Misiones", id=0):
                gr.Markdown("### Bienvenido al Simulador Pedagógico")
                gr.Markdown("Selecciona un modo para comenzar el aprendizaje guiado o experimentación libre.")
                with gr.Row():
                    btn_m1 = gr.Button("🎯 Misión 1: Atenuación", variant="primary")
                    btn_free = gr.Button("🔬 Modo Libre", variant="secondary")

            # TAB 2: Modo Guiado
            with gr.TabItem("🎯 Modo Guiado", id=1):
                gr.Markdown("### Misión Activa: Control de Atenuación y Frecuencia")
                with gr.Row():
                    with gr.Column(scale=1):
                        f0_mhz = gr.Slider(1.0, 15.0, value=5.0, label="Frecuencia f0 [MHz]")
                        mu_db = gr.Slider(0.1, 2.0, value=0.5, label="Atenuación [dB/cm/MHz]")
                        btn_run_m1 = gr.Button("Ejecutar Simulación", variant="primary")
                        btn_reset_m1 = gr.Button("Resetear")

                    with gr.Column(scale=2):
                        plot_output_m1 = gr.Plot(label="Resultado B-Mode")
                        ia_feedback = gr.Textbox(label="💡 Panel Pedagógico IA", interactive=False)

            # TAB 3: Modo Libre
            with gr.TabItem("🔬 Modo Libre", id=2):
                gr.Markdown("### Modo Avanzado de Experimentación")
                gr.Markdown("Acceso completo a todos los parámetros del transductor y tejido.")
                btn_home = gr.Button("Volver al Inicio")

        # Lógica de Navegación y Callbacks
        def run_mission_1(f0, mu):
            phantom = generate_phantom(128, 128, 40.0, 50.0, 1540, 1000, 1600, 1050, 0, 25, 5, 1500, 980, 0, 10, 3)
            res = simulate_level1_continuous(phantom, 64, 32, f0, 2.5, 16, 0.3, 0.25, 10.0, 20.0, mu, 1.0, 40.0, "linear")
            fig = plot_bmode(res["B"], phantom["x_mm"], phantom["z_mm"])
            feedback = f"Simulación completada a {f0} MHz. Nota cómo mayor frecuencia genera mayor atenuación en profundidad."
            return fig, feedback

        btn_run_m1.click(run_mission_1, inputs=[f0_mhz, mu_db], outputs=[plot_output_m1, ia_feedback])
        btn_m1.click(lambda: gr.Tabs(selected=1), None, tabs)
        btn_free.click(lambda: gr.Tabs(selected=2), None, tabs)
        btn_home.click(lambda: gr.Tabs(selected=0), None, tabs)

    return demo

if __name__ == "__main__":
    app = build_gui()
    app.launch()

Writing src/ui_gradio.py


In [14]:
import importlib
import src.ui_gradio
importlib.reload(src.ui_gradio)

app = src.ui_gradio.build_gui()
app.launch(inline=True, share=False)


/content/src/ui_gradio.py:6: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Simulador Ultrasonido 2D") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [15]:
%%writefile src/ui_gradio.py
import gradio as gr
from src.simulation_core import generate_phantom, simulate_level1_continuous
from src.plots import plot_bmode

def build_gui():
    with gr.Blocks() as demo:
        gr.Markdown("# 🩺 Simulador de Ultrasonido 2D — IALAB")

        with gr.Tabs() as tabs:
            # TAB 1: Portada
            with gr.TabItem("🏠 Portada & Misiones", id=0):
                gr.Markdown("### Bienvenido al Simulador Pedagógico")
                gr.Markdown("Selecciona un modo para comenzar el aprendizaje guiado o experimentación libre.")
                with gr.Row():
                    btn_m1 = gr.Button("🎯 Misión 1: Atenuación", variant="primary")
                    btn_free = gr.Button("🔬 Modo Libre", variant="secondary")

            # TAB 2: Modo Guiado
            with gr.TabItem("🎯 Modo Guiado", id=1):
                gr.Markdown("### Misión Activa: Control de Atenuación y Frecuencia")
                with gr.Row():
                    with gr.Column(scale=1):
                        f0_mhz = gr.Slider(1.0, 15.0, value=5.0, label="Frecuencia f0 [MHz]")
                        mu_db = gr.Slider(0.1, 2.0, value=0.5, label="Atenuación [dB/cm/MHz]")
                        btn_run_m1 = gr.Button("Ejecutar Simulación", variant="primary")
                        btn_reset_m1 = gr.Button("Resetear")

                    with gr.Column(scale=2):
                        plot_output_m1 = gr.Plot(label="Resultado B-Mode")
                        ia_feedback = gr.Textbox(label="💡 Panel Pedagógico IA", interactive=False)

            # TAB 3: Modo Libre
            with gr.TabItem("🔬 Modo Libre", id=2):
                gr.Markdown("### Modo Avanzado de Experimentación")
                gr.Markdown("Acceso completo a todos los parámetros del transductor y tejido.")
                btn_home = gr.Button("Volver al Inicio")

        # Lógica de Navegación y Callbacks
        def run_mission_1(f0, mu):
            phantom = generate_phantom(128, 128, 40.0, 50.0, 1540, 1000, 1600, 1050, 0, 25, 5, 1500, 980, 0, 10, 3)
            res = simulate_level1_continuous(phantom, 64, 32, f0, 2.5, 16, 0.3, 0.25, 10.0, 20.0, mu, 1.0, 40.0, "linear")
            fig = plot_bmode(res["B"], phantom["x_mm"], phantom["z_mm"])
            feedback = f"Simulación completada a {f0} MHz. Nota cómo mayor frecuencia genera mayor atenuación en profundidad."
            return fig, feedback

        btn_run_m1.click(run_mission_1, inputs=[f0_mhz, mu_db], outputs=[plot_output_m1, ia_feedback])
        btn_m1.click(lambda: gr.Tabs(selected=1), None, tabs)
        btn_free.click(lambda: gr.Tabs(selected=2), None, tabs)
        btn_home.click(lambda: gr.Tabs(selected=0), None, tabs)

    return demo

if __name__ == "__main__":
    app = build_gui()
    app.launch(theme=gr.themes.Soft())

Overwriting src/ui_gradio.py


In [16]:
import gradio as gr
import importlib
import src.ui_gradio

# Liberar puertos bloqueados
gr.close_all()

importlib.reload(src.ui_gradio)

app = src.ui_gradio.build_gui()
app.launch(inline=True, share=True, theme=gr.themes.Soft())

Closing server running on port: 7860
Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d09da46b0b19014bd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
